In [26]:
import os
import pandas as pd
import numpy as np

In [2]:
# only works on bigquery kernel, which I can't get working!

#google cloud big query libaries
#from google.cloud import bigquery
#from google.oauth2 import service_account

#key_path = r"C:\Users\cday\streetlight-temp-analysis-1aa223143ab5.json"

#credentials = service_account.Credentials.from_service_account_file(
#    key_path, scopes=["https://www.googleapis.com/auth/cloud-platform"],
#)

#client = bigquery.Client(credentials=credentials, project=credentials.project_id,)

In [ ]:
#directories
working_directory   = os.getcwd()
data_folder         = os.path.join(working_directory, "data"        )
intermediate_folder = os.path.join(working_directory, "intermediate")
results_folder      = os.path.join(working_directory, "results"     )
print(working_directory)

In [ ]:
#inputs
collegeDataOD_path         = os.path.join(working_directory, "intermediate/dfCollegeDataOD.csv")
college_to_SL_COTAZID_path = os.path.join(working_directory, "data/College_to_SL_COTAZID.csv")

dfCollegeDataOD = pd.read_csv(collegeDataOD_path)
College_to_SL_COTAZID = pd.read_csv(college_to_SL_COTAZID_path)
#dfCollegeDataOD
#College_to_SL_COTAZID

In [ ]:
#get origin college
dfCollegeDataOD_taz = pd.merge(dfCollegeDataOD, College_to_SL_COTAZID, how = 'left', left_on=["origin_zone_name"],right_on=["SL_COTAZID"])
dfCollegeDataOD_taz.drop('SL_COTAZID', axis=1, inplace=True)
dfCollegeDataOD_taz.rename(columns={'COLLEGE':'O_COLLEGE'}, inplace=True)
#get destination college
dfCollegeDataOD_taz = pd.merge(dfCollegeDataOD_taz, College_to_SL_COTAZID, how = 'left', left_on=["destination_zone_name"],right_on=["SL_COTAZID"])
dfCollegeDataOD_taz.drop('SL_COTAZID', axis=1, inplace=True)
dfCollegeDataOD_taz.rename(columns={'COLLEGE':'D_COLLEGE'}, inplace=True)

dfCollegeDataOD_taz

In [43]:
#bill's way of r case when in python
#dfPeriods = pd.DataFrame([
#    ["1: Early AM (12am-6am)", "EV"],   
#    ["2: Peak AM (6am-9am)"  , "AM"],   
#    ["3: Mid-Day (9am-3pm)"  , "MD"],  
#    ["4: Peak PM (3pm-6pm)"  , "PM"],  
#    ["5: Late PM (6pm-12am)" , "EV"],
#    ["0: All Day (12am-12am)", "DY"]]
#    ,columns=('day_part','period'))

#dfCollegeDataOD_taz_wperiod = pd.DataFrame.merge(dfCollegeDataOD_taz,dfPeriods,on='day_part')
#dfCollegeDataOD_taz_wperiod

In [ ]:
# r case when in python: https://stackoverflow.com/questions/54653356/case-when-function-from-r-to-python
conditions = [
    dfCollegeDataOD_taz["day_part"].eq("1: Early AM (12am-6am)"),
    dfCollegeDataOD_taz["day_part"].eq("2: Peak AM (6am-9am)"),
    dfCollegeDataOD_taz["day_part"].eq("3: Mid-Day (9am-3pm)"),
    dfCollegeDataOD_taz["day_part"].eq("4: Peak PM (3pm-6pm)"),
    dfCollegeDataOD_taz["day_part"].eq("5: Late PM (6pm-12am)"),
    dfCollegeDataOD_taz["day_part"].eq("0: All Day (12am-12am)"),
]
choices=["EV", "AM", "MD", "PM", "EV", "DY"]
dfCollegeDataOD_taz["period"] = np.select(conditions,choices)
dfCollegeDataOD_taz

In [37]:
#dfCollegeDataOD_filtered = dfCollegeDataOD_taz.query('period != `DT`')
dfCollegeDataOD_filtered = dfCollegeDataOD_taz[dfCollegeDataOD_taz['period'] != 'DT']

In [38]:
dfCollegeDataOD_filtered

,origin_zone_name,destination_zone_name,mode_of_travel,day_type,day_part,data_period,o_d_traffic_sample_trip_counts,o_d_traffic_calibrated_trip_volume,O_COLLEGE,D_COLLEGE,period
0,490763_2,490723_1,All Vehicles - StL All Vehicles Sample Trip Co...,2: Weekend Day (Sa-Su),3: Mid-Day (9am-3pm),2. Sep-Nov,13,14.500000,NaN,BYU,MD
1,350426_0,350451_0,All Vehicles - StL All Vehicles Sample Trip Co...,2: Weekend Day (Sa-Su),3: Mid-Day (9am-3pm),2. Sep-Nov,13,14.500000,NaN,NaN,MD
3,350482_0,350672_0,All Vehicles - StL All Vehicles Sample Trip Co...,2: Weekend Day (Sa-Su),3: Mid-Day (9am-3pm),2. Sep-Nov,13,14.500000,NaN,NaN,MD
7,350060_1,350090_0,All Vehicles - StL All Vehicles Sample Trip Co...,1: Weekday (Tu-Th),3: Mid-Day (9am-3pm),2. Sep-Nov,39,29.000000,NaN,NaN,MD
8,350479_0,350060_1,All Vehicles - StL All Vehicles Sample Trip Co...,0: All Days (Mo-Su),3: Mid-Day (9am-3pm),2. Sep-Nov,91,29.000000,NaN,NaN,MD
...,...,...,...,...,...,...,...,...,...,...,...
950314,110112_0,110012_0,All Vehicles - StL All Vehicles Sample Trip Co...,1: Weekday (Tu-Th),4: Peak PM (3pm-6pm),2. Sep-Nov,3,2.230769,WSU_DAVIS,NaN,PM
950315,490723_2,490777_3,All Vehicles - StL All Vehicles Sample Trip Co...,1: Weekday (Tu-Th),5: Late PM (6pm-12am),2. Sep-Nov,3,2.230769,BYU,NaN,EV
950320,570417_0,570284_6,All Vehicles - StL All Vehicles Sample Trip Co...,1: Weekday (Tu-Th),3: Mid-Day (9am-3pm),2. Sep-Nov,12,8.923077,NaN,WSU_MAIN,MD
950321,43084_0,350060_1,All Vehicles - StL All Vehicles Sample Trip Co...,1: Weekday (Tu-Th),3: Mid-Day (9am-3pm),2. Sep-Nov,12,8.923077,NaN,NaN,MD
